In [ ]:
# Pandas kutubxonasini import qilamiz — jadval ko'rinishidagi ma'lumotlar bilan ishlash uchun
import pandas as pd

# Matplotlib — grafik va diagrammalar chizish uchun asosiy kutubxona
import matplotlib.pyplot as plt

# Seaborn — matplotlibga asoslangan, chiroyli statistik grafiklar uchun
import seaborn as sns

# CSV faylni o'qib, DataFrame (jadval) ko'rinishiga o'tkazamiz
df = pd.read_csv('superstore_cleaned.csv')

# 'Order Date' ustunini matn (string) dan sana (datetime) formatiga o'tkazamiz
# Bu keyin yil, oy, kun bo'yicha guruhlash va filtrlash imkonini beradi
df['Order Date'] = pd.to_datetime(df['Order Date'])

In [ ]:
# Grafik oynasini yaratamiz — kengligi 10, balandligi 6 dyuym
plt.figure(figsize=(10, 6))

# Histogramma chizamiz:
#   df['Sales']     — Sales ustunidagi ma'lumotlar
#   bins=50         — ma'lumotlarni 50 ta ustunchaga bo'lamiz (qancha ko'p = shuncha batafsil)
#   color           — ustunchalar rangi
#   edgecolor       — ustunchalar o'rtasidagi chegara rangi
plt.hist(df['Sales'], bins=50, color='steelblue', edgecolor='black')

# Grafik sarlavhasi
plt.title('Savdo taqsimoti')

# X o'qi belgisi — gorizontal o'q (savdo miqdori)
plt.xlabel('Savdo ($)')

# Y o'qi belgisi — vertikal o'q (takrorlanish soni)
plt.ylabel('Chastota')

# Grafikni ekranda ko'rsatamiz
plt.show()

In [ ]:
# df ni 'Category' ustuni bo'yicha guruhlaymiz

cat_sales = df.groupby('Category')['Sales'].sum().sort_values(ascending=False)

print(cat_sales)

In [ ]:
# Grafik oynasini yaratamiz — kengligi 8, balandligi 5 dyuym
plt.figure(figsize=(8, 5))

# cat_sales Series'ini ustunli diagrammaga (bar chart) aylantiramiz:
#   kind='bar'    — grafik turi: vertikal ustunchalar
#   color=[...]   — har bir kategoriya uchun alohida rang (HEX formatda)
#                   '#2196F3' = ko'k (Furniture)
#                   '#4CAF50' = yashil (Office Supplies)
#                   '#FF9800' = to'q sariq (Technology)
cat_sales.plot(kind='bar', color=['#2196F3', '#4CAF50', '#FF9800'])

# Grafik sarlavhasi
plt.title('Kategoriya bo\'yicha Jami Savdo')

# X o\'qi belgisi — kategoriya nomlari
plt.xlabel('Kategoriya')

# Y o\'qi belgisi — jami savdo miqdori
plt.ylabel('Jami Savdo ($)')

# X o\'qi yozuvlarini to\'g\'ri (0 daraja) holda qoldiramiz
# rotation=45 yoki 90 qilsangiz — yozuvlar qiyshayadi
plt.xticks(rotation=0)

# Grafikni ekranda ko\'rsatamiz
plt.show()

In [ ]:
# Q1 — birinchi kvartil (25% chegara)
# ya'ni ma'lumotlarning 25% shu qiymatdan PAST, 75% esa YUQORI
Q1 = df['Profit'].quantile(0.25)

# Q3 — uchinchi kvartil (75% chegara)
# ya'ni ma'lumotlarning 75% shu qiymatdan PAST, 25% esa YUQORI
Q3 = df['Profit'].quantile(0.75)

# IQR (Interquartile Range) — kvartillararo masofa
# bu Q1 va Q3 orasidagi farq bo'lib, ma'lumotlarning "normal" tarqalish oralig'ini ko'rsatadi
# outlier (g'ayritabiiy qiymat) aniqlashda ishlatiladi
IQR = Q3 - Q1

# Natijalarni 2 xona aniqlikda chiqaramiz (:.2f = 2 ta kasr raqam)
print(f"Q1: {Q1:.2f}, Q3: {Q3:.2f}, IQR: {IQR:.2f}")

In [ ]:
# Quyi chegara — bu qiymatdan PAST bo'lgan har qanday ma'lumot outlier hisoblanadi
# Formula: Q1 - 1.5 × IQR
lower_bound = Q1 - 1.5 * IQR

# Yuqori chegara — bu qiymatdan YUQORI bo'lgan har qanday ma'lumot outlier hisoblanadi
# Formula: Q3 + 1.5 × IQR
upper_bound = Q3 + 1.5 * IQR

# Chegaralarni 2 xona aniqlikda chiqaramiz
print(f"Quyi chegara: {lower_bound:.2f}, Yuqori chegara: {upper_bound:.2f}")

In [ ]:
# Outlierlarni filtrlash — ikki shartdan biri bajarilsa outlier hisoblanadi:
#   df['Profit'] < lower_bound  — quyi chegaradan PAST
#   df['Profit'] > upper_bound  — yuqori chegaradan YUQORI
#   | belgisi — "YOKI" (OR) mantig'i: ikkisidan biri to'g'ri bo'lsa yetarli
outliers = df[(df['Profit'] < lower_bound) | (df['Profit'] > upper_bound)]

# Natijani chiqaramiz:
#   len(outliers)          — outlier qatorlar soni
#   len(df)                — jami qatorlar soni
#   len(outliers)/len(df)  — outlierlar ulushi (0.03 = 3% degani)
#   :.2%                   — foiz formatida chiqarish (0.034 → 3.40%)
print(f"Outlierlar soni: {len(outliers)} ta, jami {len(df)} qatordan ({len(outliers)/len(df):.2%})")

In [ ]:

# KORRELYATSIYA MATRITSASI — ustunlar orasidagi bog'liqlikni o'lchaydi


# Faqat raqamli ustunlarni tanlaymiz — korrelyatsiya faqat sonlar uchun hisoblanadi
numerical_cols = df[['Sales', 'Quantity', 'Discount', 'Profit']]

# Har bir ustun juftligi orasidagi korrelyatsiyani hisoblaymiz
# Natija: 4x4 li matritsa (har ustun o'zi bilan 1.0, boshqalar bilan -1 dan +1 gacha)
corr_matrix = numerical_cols.corr()

# Matritsani chiqaramiz
print(corr_matrix)



# HEATMAP — korrelyatsiya matritsasini rangli ko'rinishda chizish


# Grafik oynasini yaratamiz
plt.figure(figsize=(8, 6))

# Seaborn heatmap:
#   corr_matrix   — korrelyatsiya matritsasi (yuqorida hisoblangan)
#   annot=True    — har bir katakka qiymatni yozadi (masalan: -0.32)
#   cmap='coolwarm' — rang palitrasi: ko'k (manfiy) → oq (0) → qizil (musbat)
#   center=0      — 0 qiymat oq rang bo'ladi (markaz nuqta)
#   fmt='.2f'     — qiymatlarni 2 xona aniqlikda ko'rsatadi
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', center=0, fmt='.2f')

# Grafik sarlavhasi
plt.title('Korrelyatsiya Heatmap: Savdo, Miqdor, Chegirma, Foyda')

# Grafikni ekranda ko'rsatamiz
plt.show()


# OYLIK SAVDO TENDENSIYASI — vaqt bo'yicha o'zgarishni ko'rish


# df ni 'Order Date' ustuni bo'yicha oylik ('M') kesimda guruhlaydi
# har oy uchun 'Sales' ustunini yig'adi
# Eslatma: 'Order Date' oldin pd.to_datetime() bilan sana formatiga o'tkazilgan edi
monthly_sales = df.resample('M', on='Order Date')['Sales'].sum()

# Grafik oynasini yaratamiz — keng (12) chunki ko'p oylar bo'ladi
plt.figure(figsize=(12, 6))

# Chiziqli grafik:
#   color='steelblue' — chiziq rangi
#   linewidth=2       — chiziq qalinligi (1 = ingichka, 3+ = qalin)
monthly_sales.plot(color='steelblue', linewidth=2)

# Grafik sarlavhasi va o'q belgilari
plt.title('Oylik Jami Savdo (Vaqt Bo\'yicha)')
plt.xlabel('Oy')
plt.ylabel('Jami Savdo ($)')

# Fon to'ri — grafikni o'qishni osonlashtiradi
#   True     — to'rni yoqamiz
#   alpha=0.3 — to'r shaffofligini kamaytiradi (0=ko'rinmas, 1=to'liq ko'rinadi)
plt.grid(True, alpha=0.3)

# Grafikni ekranda ko'rsatamiz
plt.show()